# NuCLS E2 · Cierre y pendientes en un cuaderno
Ejecutar **solo después de terminar la cola actual** y en la misma cuenta de Drive. Ejecutar todo y autorizar Drive. No se modifican E1 ni el cuaderno activo.

Secuencia: dependencias → referencias simples si faltan → cierre de reserva y estadística → ensembles → sanidad → Grad-CAM → tres ablaciones × cinco semillas → réplica parcial PCam → acuerdo multi-rater si hay anotaciones → inventario de salidas.

**Incluye entrenamientos NUEVOS de ablación y PCam. Puede exceder el día de entrega.** No repite corridas completas verificadas; una época interrumpida puede repetirse. No completa pilotos faltantes: exige las 65 corridas previstas antes de abrir reserva. El cuaderno no garantiza que Colab permanezca conectado y se detiene ante una marca activa o una inconsistencia, sin borrar bloqueos.

PCam queda habilitado para atender todos los bloques implementados; corre al final, después de guardar el cierre urgente. La réplica es parcial, no cumplimiento íntegro del paper guía. Revisión histológica y anotaciones ausentes no se pueden sustituir con ejecución automática. Robustez se recibe de la cola previa; este cuaderno no la inventa ni la recalcula.

**Señal para descargar hoy:** `CIERRE_URGENTE_RESPALDADO`. El ZIP urgente ya contiene el cierre y la estadística; no hace falta esperar PCam. El preflight comprueba65 fuentes antes de preparar referencias. Los modelos profundos completados no se entrenan de nuevo; la logística solo se ajusta si falta, y las ablaciones y PCam son experimentos nuevos posteriores al respaldo.


## 1. Conectar Drive y comprobar que terminó el desarrollo

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('MPLCONFIGDIR', '/content/nucls_cierre_mpl')
from google.colab import drive
MOUNT = Path('/content/drive')
if not (MOUNT / 'MyDrive').is_dir():
    if MOUNT.is_dir() and any(MOUNT.iterdir()):
        raise RuntimeError('El punto de montaje contiene archivos pero no MyDrive. No se borró nada; reconecta el entorno antes de continuar.')
    drive.mount(str(MOUNT))
DRIVE = MOUNT / 'MyDrive'
assert DRIVE.is_dir(), 'Drive no está disponible.'
for folder in ['NuCLS_E2_ADICIONALES_EXPLORATORIO', 'NuCLS_E2_TRES_NUEVOS_V1', 'NuCLS_E2_COLA_UNICA_V1', 'NuCLS_E2_CIERRE_RESERVA_V1']:
    lock = DRIVE / folder / 'EJECUCION_ACTIVA.json'
    if lock.exists():
        # Solo la marca del cierre propio y de un proceso muerto en este host podrá recuperarse dentro del programa.
        import platform
        info = json.loads(lock.read_text())
        recoverable = False
        if folder == 'NuCLS_E2_CIERRE_RESERVA_V1' and info.get('host') == platform.node():
            try: os.kill(int(info['pid']), 0)
            except ProcessLookupError: recoverable = True
        if not recoverable:
            raise RuntimeError('Existe una marca de ejecución: ' + str(lock) + '. Verifica que terminó esa sesión; no se borró ni se inició otra copia.')
print('Drive disponible. Comprobación inicial completada; aún no se evaluó reserva.')

# Verificar fuentes antes de instalar dependencias o ajustar referencias simples.
"""Inventario temprano de solo lectura; no monta Drive, abre reserva ni inicia procesos."""
from pathlib import Path
import csv,json,hashlib
BASES=['softmaxcnn_bn','densenet121','resnet50']
SIETE=['resnet18','mobilenetv3_small_100','efficientnet_b0','deit_tiny_patch16_224','swin_tiny_patch4_window7_224','mobilevit_xxs','mixer_s16_224']
TRES=['convnext_tiny','inception_v3','sequencer2d_s']
def comprobar_fuentes(drive):
 drive=Path(drive)
 if not (drive/'NuCLS_V2_DATOS.zip').is_file():raise FileNotFoundError('Falta NuCLS_V2_DATOS.zip en esta cuenta. No se inició evaluación.')
 root=drive/'NuCLS_E2_CONTINUACION_BASES_22';idx=root/'indice_validacion.csv'
 if not idx.is_file():idx=root/'final/indice_validacion.csv'
 if not idx.is_file():raise FileNotFoundError('Falta el índice de las bases en esta cuenta de Drive.')
 mapping={}
 with idx.open() as f:
  for row in csv.DictReader(f):
   key=(row['modelo'],int(row['semilla']))
   if key in mapping:raise ValueError('Índice duplicado: '+str(key))
   mapping[key]=row
 missing=[];verified=[]
 for model in BASES+SIETE+TRES:
  for seed in range(6101,6106):
   item=mapping.get((model,seed))
   folder=Path(item['ruta']) if model in BASES and item else (None if model in BASES else drive/('NuCLS_E2_TRES_NUEVOS_V1' if model in TRES else 'NuCLS_E2_ADICIONALES_EXPLORATORIO')/'piloto'/f'{model}_{seed}')
   files=['resultado.json','historia.json','predicciones_validacion.csv','mejor.pt']
   if folder is None or any(not (folder/n).is_file() or (folder/n).stat().st_size==0 for n in files):missing.append(f'{model}_{seed}');continue
   q=json.loads((folder/'resultado.json').read_text())
   if (q.get('modelo'),q.get('semilla'),q.get('estado'))!=(model,seed,'FASE_COMPLETADA_VALIDACION'):raise ValueError('Resultado incompleto o de otra corrida: '+str(folder))
   if model in BASES and item.get('resultado_sha256') and hashlib.sha256((folder/'resultado.json').read_bytes()).hexdigest()!=item['resultado_sha256']:raise ValueError('Resultado distinto del índice: '+str(folder))
   verified.append(f'{model}_{seed}')
 if missing:raise RuntimeError(f'Preflight: {len(verified)}/65 presentes. Faltan: '+', '.join(missing)+'. No se entrenó ni se abrió reserva.')
 return {'estado':'FUENTES_65_PRESENTES','corridas':len(verified),'nota':'Presencia e identidad preliminares; el cierre aún verificará hashes, historias, predicciones y checkpoints.'}

print(json.dumps(comprobar_fuentes(DRIVE),ensure_ascii=False,indent=2))


## 2. Preparar el código de cierre y sus dependencias

In [ ]:
# No se actualiza PyTorch ni se reinstala la GPU.
import importlib.util
import base64, hashlib, io, zipfile
requirements = []
for module, package in [('scipy','scipy'),('sklearn','scikit-learn'),('matplotlib','matplotlib'),('joblib','joblib')]:
    if importlib.util.find_spec(module) is None: requirements.append(package)
try: timm_version = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError: timm_version = None
if timm_version != '1.0.22': requirements.append('timm==1.0.22')
classic = DRIVE / 'NuCLS_E2_REFERENCIAS_SIMPLES' / 'resultado.json'
if classic.is_file():
    wanted = json.loads(classic.read_text()).get('sklearn')
    if wanted:
        requirements = [r for r in requirements if r != 'scikit-learn']
        try: current = importlib.metadata.version('scikit-learn')
        except importlib.metadata.PackageNotFoundError: current = None
        if current != wanted: requirements.append('scikit-learn==' + wanted)
if requirements:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + requirements, check=True)
CODIGO = Path('/content/nucls_e2_cierre_reserva_v1_codigo')
CODIGO.mkdir(exist_ok=True)
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == '49997080c0e84ecae5544d8fb4227339da81e104bdc255c0fb684c838167523a'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODIGO / name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
print('Código íntegro verificado. No se inició entrenamiento.')


## Referencias simples antes de la reserva
Reutiliza la logística guardada cuando sus archivos completos existen. Si faltan, ajusta solo con train y selecciona por validación.


In [ ]:
BASELINE_SOURCE='"""Referencias exploratorias: mayoría de train y regresión logística con cuatro C."""\nimport argparse,sys,time,json\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'modelos\'))\nimport numpy as np\nfrom PIL import Image\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nimport sklearn\nimport core_v2 as core\n\ndef features(root,rows):\n xs=[];ys=[]\n for r in rows:\n  with Image.open(root/r[\'archivo\']) as im:a=np.asarray(im.convert(\'RGB\'),dtype=np.float32)/255\n  # Medias por celdas de 8x8 y 16 bins RGB: representación fija, no entrenada.\n  pooled=a.reshape(16,8,16,8,3).mean((1,3)).ravel()\n  hist=np.concatenate([np.histogram(a[:,:,c],bins=16,range=(0,1),density=False)[0]/(128*128) for c in range(3)])\n  xs.append(np.r_[pooled,hist]);ys.append(core.CLASSES.index(r[\'clase\']))\n return np.asarray(xs),np.asarray(ys)\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--datos\',type=Path,required=True);p.add_argument(\'--salida\',type=Path,required=True);a=p.parse_args()\n if a.salida.name!=\'NuCLS_E2_REFERENCIAS_SIMPLES\':raise ValueError(\'Carpeta exclusiva requerida\')\n if a.salida.is_symlink():raise ValueError(\'Salida enlazada no permitida\')\n a.salida.mkdir(parents=True,exist_ok=True)\n if (a.salida/\'resultado.json\').exists() and all((a.salida/f).is_file() for f in [\'logistica_predicciones.csv\',\'mayoritaria_predicciones.csv\',\'logistica.joblib\']):print(\'Referencias ya calculadas\');return\n groups=core.read_data(a.datos);x,y=features(a.datos,groups[\'train\']);v,vy=features(a.datos,groups[\'validation\'])\n scaler=StandardScaler().fit(x);x=scaler.transform(x);v=scaler.transform(v);trials=[];best=None\n for c in [.001,.01,.1,1.]:\n  start=time.perf_counter();m=LogisticRegression(C=c,max_iter=2000,solver=\'lbfgs\',random_state=7001);m.fit(x,y);prob=m.predict_proba(v)\n  if int(m.n_iter_.max())>=2000:raise RuntimeError(\'La regresión no convergió; revisar antes de comparar\')\n  met=core.scores(vy,prob.argmax(1));trials.append(dict(C=c,segundos=time.perf_counter()-start,**met));print(\'Logística\',c,met[\'macro_f1\'],flush=True)\n  if best is None or met[\'macro_f1\']>best[0]:best=(met[\'macro_f1\'],prob,c,m)\n majority=int(np.bincount(y,minlength=3).argmax());trivial=core.scores(vy,np.full(len(vy),majority))\n result=dict(estado=\'EXPLORATORIO_VALIDACION\',sklearn=sklearn.__version__,busqueda_logistica=trials,C_elegido=best[2],mayoritaria=trivial,clase_mayoritaria=core.CLASSES[majority],nota=\'Deterministas: no se fabrican cinco repeticiones idénticas. Escalador ajustado únicamente a train.\')\n for name,prob in [(\'logistica\',best[1]),(\'mayoritaria\',np.tile(np.eye(3)[majority],(len(vy),1)))]:\n  records=[dict(paciente=r[\'paciente\'],archivo=r[\'archivo\'],clase_real=r[\'clase\'],clase_predicha=core.CLASSES[int(prob[i].argmax())],p_tumor=float(prob[i,0]),p_linfocito=float(prob[i,1]),p_estroma=float(prob[i,2])) for i,r in enumerate(groups[\'validation\'])];core.csv_write(a.salida/(name+\'_predicciones.csv\'),records)\n import joblib\n joblib.dump({\'scaler\':scaler,\'model\':best[3],\'clases\':core.CLASSES},a.salida/\'logistica.joblib\')\n core.save_json(a.salida/\'resultado.json\',result)\n print(core.archive(a.salida))\nif __name__==\'__main__\':main()\n'
# Referencias exclusivamente de train/validación antes de abrir reserva.
(CODIGO/'baselines_simples.py').write_text(BASELINE_SOURCE)
preparar = '''
import sys,json,zipfile,hashlib,csv
from pathlib import Path
import cierre_unico as f
D=Path(sys.argv[1]); f.check_idle(D)
source=D/'NuCLS_V2_DATOS.zip'; target=Path('/content/nucls_e2_referencias_datos');target.mkdir(exist_ok=True)
with zipfile.ZipFile(source) as z:
 raw=z.read('manifiesto.csv');assert hashlib.sha256(raw).hexdigest()==f.core.EXPECTED
 rows=list(csv.DictReader(raw.decode().splitlines()));rows=[r for r in rows if r['particion'] in ['train','validation'] and r['estado']=='incluida']
 f.extract(z,target,rows);(target/'manifiesto.csv').write_bytes(raw)
'''
subprocess.run([sys.executable,'-c',preparar,str(DRIVE)],cwd=CODIGO,check=True)
subprocess.run([sys.executable,'-u',str(CODIGO/'baselines_simples.py'),'--datos','/content/nucls_e2_referencias_datos','--salida',str(DRIVE/'NuCLS_E2_REFERENCIAS_SIMPLES')],check=True)


## 3. Ejecutar el cierre completo

Primero verifica las 65 corridas y reproduce validación. Luego congela las entradas, evalúa reserva, calcula la brecha con inferencia sobre train y genera la estadística. Puede tardar varias horas según GPU; no contiene entrenamiento.

Cada cierre de corrida genera un respaldo. No vuelvas a pulsar ejecutar mientras esta celda esté activa. Si se interrumpe, vuelve a ejecutar este mismo cuaderno; una marca de otra sesión exige verificarla antes de continuar.


In [ ]:
import datetime, signal
SALIDA = DRIVE / 'NuCLS_E2_CIERRE_RESERVA_V1'
assert not SALIDA.is_symlink(), 'La salida no puede ser un enlace a otra carpeta.'
SALIDA.mkdir(exist_ok=True)
log = SALIDA / ('log_cierre_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.txt')
with log.open('w') as registro:
    proceso = subprocess.Popen([sys.executable, '-u', str(CODIGO / 'cierre_unico.py'), '--drive', str(DRIVE)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for linea in proceso.stdout:
            print(linea, end='', flush=True)
            registro.write(linea); registro.flush()
        retorno = proceso.wait()
    except KeyboardInterrupt:
        proceso.send_signal(signal.SIGINT)
        try: proceso.wait(timeout=60)
        except subprocess.TimeoutExpired:
            proceso.terminate(); proceso.wait(timeout=30)
        raise
print('Log:', log)
if retorno != 0:
    raise RuntimeError('El cierre se detuvo con evidencia conservada. Lee el diagnóstico anterior; no borres resultados ni bloqueos y no cambies de cuaderno.')


## 4. Recoger el informe y las evidencias

In [ ]:
estado = SALIDA / 'estado.json'
assert estado.is_file(), 'No hay cierre registrado.'
resultado = json.loads(estado.read_text())
print(json.dumps(resultado, ensure_ascii=False, indent=2))
assert resultado.get('estado') == 'CIERRE_RESERVA_Y_ESTADISTICA_COMPLETADO', 'El cierre aún no terminó.'
# Refrescar ZIP una vez cerrado el registro de la celda anterior.
import zipfile
zip_final = DRIVE / 'NuCLS_E2_CIERRE_RESERVA_V1_RESULTADOS.zip'
temporal = zip_final.with_suffix('.tmp')
with zipfile.ZipFile(temporal, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SALIDA.rglob('*')):
        if p.is_file() and p.name != 'EJECUCION_ACTIVA.json' and p.suffix in ['.json','.csv','.npz','.html','.png','.txt','.py']:
            z.write(p, str(p.relative_to(SALIDA)))
temporal.replace(zip_final)
print('ZIP FINAL:', zip_final)
print('INFORME:', SALIDA / 'informe.html')
print('Guarda este notebook con salidas: Archivo → Descargar → Descargar .ipynb.')
print('Descarga el ZIP desde Drive. Extrae todo antes de abrir informe.html.')

# Punto de entrega persistente ANTES de los experimentos largos.
"""Conserva un cierre íntegro antes de los experimentos largos; solo usa archivos locales."""
import hashlib,json,zipfile,shutil,os
from pathlib import Path

def respaldar(zip_cierre,drive):
    source=Path(zip_cierre);drive=Path(drive)
    with zipfile.ZipFile(source) as z:
        if z.testzip() is not None:raise ValueError('ZIP de cierre dañado')
        state=json.loads(z.read('estado.json'))
        if state.get('estado')!='CIERRE_RESERVA_Y_ESTADISTICA_COMPLETADO' or state.get('corridas')!=65:
            raise ValueError('El cierre no acredita 65 corridas completas')
        for name in ['entradas_fijadas.json','congelamiento_previo_reserva.json','resumen_reserva.csv','contrastes_vs_resnet50.csv','informe.html','benchmark_reserva.png','diferencia_critica.png']:
            if name not in z.namelist():raise ValueError('Salida de cierre ausente: '+name)
    digest=hashlib.sha256(source.read_bytes()).hexdigest()
    folder=drive/'NuCLS_E2_ENTREGA_URGENTE'
    if folder.is_symlink():raise ValueError('Destino enlazado no permitido')
    folder.mkdir(exist_ok=True)
    target=folder/('CIERRE_65_'+digest[:16]+'.zip')
    if target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest()!=digest:raise ValueError('Respaldo existente alterado; no se sobrescribe')
    else:
        temp=target.with_suffix('.tmp')
        with temp.open('xb') as out,source.open('rb') as inp:
            shutil.copyfileobj(inp,out);out.flush();os.fsync(out.fileno())
        if hashlib.sha256(temp.read_bytes()).hexdigest()!=digest:raise ValueError('Copia no coincide')
        temp.replace(target)
    receipt=folder/('RECIBO_'+digest[:16]+'.json')
    data={'estado':'CIERRE_URGENTE_RESPALDADO','zip':target.name,'sha256':digest,'corridas':65,
          'alcance':'Cierre y estadística; no certifica todos los requisitos ni los experimentos posteriores',
          'siguiente':'Continuar experimentos largos; este respaldo queda disponible para descargar'}
    if receipt.exists():
        if json.loads(receipt.read_text())!=data:raise ValueError('Recibo existente no coincide')
    else:
        with receipt.open('x') as f:json.dump(data,f,ensure_ascii=False,indent=2)
    return target,receipt

ZIP_URGENTE, RECIBO_URGENTE = respaldar(zip_final, DRIVE)
print('\nCIERRE_URGENTE_RESPALDADO — YA PUEDES DESCARGARLO DESDE DRIVE')
print('ZIP:', ZIP_URGENTE)
print('Recibo:', RECIBO_URGENTE)
print('El cuaderno continúa automáticamente. No detener para descargar desde otra pestaña de Drive.')


## Experimentos largos: después del respaldo urgente
La celda anterior debe mostrar `CIERRE_URGENTE_RESPALDADO`. Desde otra pestaña de Drive se puede descargar el ZIP de `NuCLS_E2_ENTREGA_URGENTE` mientras continúan estos experimentos. Si falla el respaldo, no comienza esta secuencia. El ZIP urgente no se actualiza con los experimentos posteriores.


In [ ]:
assert 'ZIP_URGENTE' in globals() and ZIP_URGENTE.is_file(), 'Ejecutar primero el cierre y su respaldo urgente.'
assert hashlib.sha256(ZIP_URGENTE.read_bytes()).hexdigest()==json.loads(RECIBO_URGENTE.read_text())['sha256'], 'El respaldo cambió; no iniciar experimentos.'
from pathlib import Path
import os,sys,json,subprocess,zipfile,io,base64,hashlib
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
EJECUTAR_PCAM=True
CERRADO=DRIVE/'NuCLS_E2_CIERRE_RESERVA_V1'
for folder in ['NuCLS_E2_ADICIONALES_EXPLORATORIO','NuCLS_E2_TRES_NUEVOS_V1','NuCLS_E2_COLA_UNICA_V1','NuCLS_E2_CIERRE_RESERVA_V1','NuCLS_E2_EXPERIMENTOS_POST_V1','NuCLS_E2_REPLICA_PCAM_V1']:
    if (DRIVE/folder/'EJECUCION_ACTIVA.json').exists():
        raise RuntimeError('Hay una marca de ejecución: '+folder+'. No se modificó. Esperar a que termine y verificar la sesión.')
assert (CERRADO/'estado.json').is_file(), 'Falta ejecutar el cuaderno de cierre en esta cuenta de Drive.'
assert json.loads((CERRADO/'estado.json').read_text())['estado']=='CIERRE_RESERVA_Y_ESTADISTICA_COMPLETADO', 'El cierre de reserva aún no está completo.'
assert (DRIVE/'NuCLS_V2_DATOS.zip').is_file(), 'Falta el ZIP original de datos en esta cuenta.'
print('Fuentes encontradas. No se inició entrenamiento.')


In [ ]:
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='0850839ac32a6d7fcce3e9608529fc9d7e5f9b483a6a4ba25d6dc1314c4bdd78'
CODIGO=Path('/content/nucls_e2_post_v1_codigo');CODIGO.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target=(CODIGO/name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.write_bytes(z.read(name))
subprocess.run([sys.executable,'-m','pip','install','timm==1.0.22','h5py'],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_transformaciones.py')],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_estadistica.py')],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_reanudacion.py')],check=True)
print('Código verificado y pruebas aprobadas.')


In [ ]:
import datetime
SALIDA=DRIVE/'NuCLS_E2_EXPERIMENTOS_POST_V1';SALIDA.mkdir(exist_ok=True)
log=SALIDA/('log_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.txt')
def ejecutar(script,*args):
    command=[sys.executable,'-u',str(CODIGO/script),*map(str,args)]
    with log.open('a') as f:
        f.write('\n'+repr(command)+'\n');f.flush()
        proceso=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in proceso.stdout:
                print(line,end='',flush=True);f.write(line);f.flush()
            retorno=proceso.wait()
        except BaseException:
            proceso.terminate();proceso.wait();raise
    if retorno:raise RuntimeError(f'{script} se detuvo con código {retorno}. Se conservó lo guardado. Ver '+str(log))
ejecutar('ensembles.py','--cierre',CERRADO,'--salida',SALIDA/'ensembles')
ejecutar('experimentos.py','--drive',DRIVE)
if EJECUTAR_PCAM:
    ejecutar('replica_pcam.py','--datos','/content/pcam_oficial','--salida',DRIVE/'NuCLS_E2_REPLICA_PCAM_V1')
print('ADICIONALES_SOLICITADOS_COMPLETADOS')
print('Resultados:', SALIDA.parent/(SALIDA.name+'_RESULTADOS.zip'))
print('Guardar este notebook con salidas: Archivo → Descargar → Descargar .ipynb')


In [ ]:
# Anotaciones externas: solo se analiza un CSV real con esquema documentado.
ANOTACIONES=DRIVE/'NuCLS_E2_ANOTACIONES_MULTIRATER.csv'
if ANOTACIONES.is_file():
    ejecutar('techo_multirater.py','--anotaciones',ANOTACIONES,'--salida',SALIDA/'techo_multirater')
else:
    print('PENDIENTE EXTERNO: anotaciones multi-rater; no se estimó un techo.')
# Inventario final. No convierte preparación o ausencia de datos en cumplimiento.
fuentes=['NuCLS_E2_CIERRE_RESERVA_V1_RESULTADOS.zip','NuCLS_E2_EXPERIMENTOS_POST_V1_RESULTADOS.zip']
for nombre in fuentes:
    p=DRIVE/nombre
    print('DISPONIBLE' if p.is_file() else 'PENDIENTE',p)
print('PCam:',DRIVE/'NuCLS_E2_REPLICA_PCAM_V1')
print('Revisar salidas y guardar ESTE notebook con salidas. Las revisiones histológicas siguen siendo humanas.')
